## Merged Features (Cross-Table)

Now combining cleaned and feature-enriched tables to create master datasets and derive features that require information from multiple sources.

In [1]:
import pandas as pd

In [2]:
orders = pd.read_csv("../data/feature_engineered/orders_fe.csv")
order_items = pd.read_csv("../data/feature_engineered/order_items_fe.csv")
customers = pd.read_csv("../data/feature_engineered/customers_fe.csv")
sellers = pd.read_csv("../data/feature_engineered/sellers_fe.csv")
products = pd.read_csv("../data/feature_engineered/products_fe.csv")
order_payments = pd.read_csv("../data/feature_engineered/order_payments_fe.csv")
order_reviews = pd.read_csv("../data/feature_engineered/order_reviews_fe.csv")
closed_deals = pd.read_csv("../data/feature_engineered/closed_deals_fe.csv")
marketing_leads = pd.read_csv("../data/feature_engineered/marketing_leads_fe.csv")

product_category_name_translations = pd.read_csv(
    "../data/processed/category_translation_clean.csv"
)
geo_locations = pd.read_csv("../data/processed/geolocation_clean.csv")

Before directly performing a merge, I verify the grains and keys of the tables I have, because this is fundamental to a successful merge. This ensures that the feature engineer tables I previously saved are also read correctly.

In [3]:
tables = {
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
    "sellers": sellers,
    "products": products,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "closed_deals": closed_deals,
    "marketing_leads": marketing_leads,
    "geo_locations": geo_locations,
    "category_translation": product_category_name_translations
}

for name, df in tables.items():
    print(f"{name}: {df.shape}")

orders: (99441, 20)
order_items: (112650, 16)
customers: (99441, 7)
sellers: (3095, 6)
products: (32951, 12)
order_payments: (103886, 5)
order_reviews: (98673, 10)
closed_deals: (842, 18)
marketing_leads: (8000, 8)
geo_locations: (1000163, 5)
category_translation: (71, 2)


I also check if the keys are truly unique. Apart from that, I don't expect order_items, order_payments, and order_reviews to be unique because they are in different grains.

In [4]:
print("orders order_id unique:", orders["order_id"].is_unique)
print("customers customer_id unique:", customers["customer_id"].is_unique)
print("sellers seller_id unique:", sellers["seller_id"].is_unique)
print("products product_id unique:", products["product_id"].is_unique)
print("marketing_leads mql_id unique:", marketing_leads["mql_id"].is_unique)
print("closed_deals mql_id unique:", closed_deals["mql_id"].is_unique)

orders order_id unique: True
customers customer_id unique: True
sellers seller_id unique: True
products product_id unique: True
marketing_leads mql_id unique: True
closed_deals mql_id unique: True


In [5]:
orders_before = len(orders)
customers_before = len(customers)

print("Orders:", orders_before)
print("Customers:", customers_before)

Orders: 99441
Customers: 99441


In [6]:
print(
    "Orders with customer_id:",
    orders["customer_id"].notna().sum()
)

print(
    "Unique customer_ids in orders:",
    orders["customer_id"].nunique()
)

Orders with customer_id: 99441
Unique customer_ids in orders: 99441


In [7]:
df_master = orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

In [8]:
print("Before:", len(orders))
print("After:", len(df_master))

Before: 99441
After: 99441


After the orders and customers tables were merged, the number of columns in the df_master table increased, as expected.

In [9]:
print(df_master.shape)

(99441, 26)


Now, I will also bind the necessary properties related to the "order_items" table to this df_master variable. If I used a different variable name for each merge operation, there would be many tables and it would look very messy. So I continue via "df_master".
First, I examine the order_items table.

In [10]:
order_items.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'item_total_cost',
       'freight_ratio', 'items_in_order', 'order_total_price',
       'order_total_freight', 'order_total_cost', 'sellers_in_order',
       'is_free_shipping', 'average_item_price'],
      dtype='str')

But in this case, directly merging the `order_items` table with `df_master` is an incorrect approach. If an order has 3 items, the row in `df_master` will be repeated 3 times. This results in order-level information, such as customer information and order details, being repeated row by row. This corrupts the `grain` of `df_master` and causes the "1 row = 1 order" property to be lost.

In [11]:
order_items.shape

(112650, 16)

Therefore, instead of directly linking the `order_items` table, I'm creating an `order_item_summary` variable. I'm also creating some of the features already present in the `order_items` table here. This is because I want to convert the `item-grain` data into an `order-grain` summary table. This also reduces the number of rows.

In [12]:
order_item_summary = (
    order_items
    .groupby("order_id")
    .agg(
        items_in_order=("order_item_id", "count"),
        order_total_price=("price", "sum"),
        order_total_freight=("freight_value", "sum"),
        order_total_cost=("item_total_cost", "sum"),
        sellers_in_order=("seller_id", "nunique"),
        is_free_shipping=("freight_value", lambda x: (x == 0).all()),
        average_item_price=("price", "mean")
    )
    .reset_index()
)

In [13]:
print(order_item_summary.shape)
print(order_item_summary["order_id"].nunique())

(98666, 8)
98666


After the merge operation, the properties from the order_item_summary table are also added to df_master, and the number of columns naturally increases.

In [14]:
df_master = df_master.merge(
    order_item_summary,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print(df_master.shape)
print(df_master["order_id"].nunique())


(99441, 33)
99441


After merging the `order_item_summary` with `df_master`, I checked the newly added order-level features for missing values.

Since `df_master` was merged using a **left join**, these missing values indicate that 775 orders in `df_master` do not have a matching record in `order_item_summary`.

This is important to investigate later, but the missing values should not be filled with arbitrary values at this stage. First, the remaining order-level tables will be integrated and the reasons for these missing records will be evaluated.

In [15]:
df_master[
    [
        "items_in_order",
        "order_total_price",
        "order_total_freight",
        "order_total_cost",
        "sellers_in_order",
        "is_free_shipping",
        "average_item_price"
    ]
].isna().sum()

items_in_order         775
order_total_price      775
order_total_freight    775
order_total_cost       775
sellers_in_order       775
is_free_shipping       775
average_item_price     775
dtype: int64

Similarly, I'm examining the order_payments table. Looking at the row counts and the number of unique order_id entries, I see that there can be multiple payment records for a single order. In other words, the grain of this table is "1 row = 1 payment record". Again, I need to make this table compliant with df_master.

In [16]:
print(order_payments.shape)
print(order_payments["order_id"].nunique())

(103886, 5)
99440


In [17]:
payment_summary = (
    order_payments
    .groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_count=("payment_sequential", "count")
    )
    .reset_index()
)

In [18]:
print(payment_summary.shape)
print(payment_summary["order_id"].nunique())

(99440, 3)
99440


In [19]:
df_master = df_master.merge(
    payment_summary,
    on="order_id",
    how="left",
    validate="one_to_one"
)

The `payment_summary` table has 3 columns, one of which is `order_id` and is shared with `df_master`. Therefore, the number of columns in the `df_master` table will be updated to 33 + 2.

In [20]:
print(df_master.shape)
print(df_master["order_id"].nunique())

(99441, 35)
99441


The `order_reviews` table has a grain of **1 row = 1 order**, as `order_id` is unique after the cleaning process. Therefore, unlike `order_items` or `order_payments`, no additional aggregation is required before merging it with `df_master`.
Since `df_master` also has a grain of **1 row = 1 order**, the two tables can be safely merged using a one-to-one relationship.

In [21]:
print(order_reviews.shape)
print(order_reviews["order_id"].nunique())

(98673, 10)
98673


In [22]:
df_master = df_master.merge(
    order_reviews,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print(df_master.shape)
print(df_master["order_id"].nunique())

(99441, 44)
99441


After merging the review data, the review-related columns were checked for missing values.

There are 768 orders without a corresponding review record. These missing values are expected because not every order has a corresponding review.

The comment-related columns have substantially more missing values because a customer may have submitted a review without leaving a written comment. Therefore, missing comment text does not necessarily mean that the review itself is missing.


In [23]:
df_master[
    [
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp",
        "response_time_hours",
        "has_comment",
        "review_length"
    ]
].isna().sum()

review_score                 768
review_comment_title       87889
review_comment_message     58665
review_creation_date         768
review_answer_timestamp      768
response_time_hours          768
has_comment                  768
review_length                768
dtype: int64

The `closed_deals` table is related to `marketing_leads` through `mql_id`. All 842 `mql_id` values in `closed_deals` have a corresponding record in `marketing_leads`.

Therefore, each closed deal can be linked to exactly one marketing lead, while not every marketing lead has a closed deal.

In [24]:
print(marketing_leads["mql_id"].nunique())
print(closed_deals["mql_id"].nunique())

print(closed_deals["mql_id"].isin(marketing_leads["mql_id"]).sum())

8000
842
842


In [25]:
marketing_leads["mql_id"].duplicated().sum()

np.int64(0)

In [26]:
closed_deals["mql_id"].duplicated().sum()

np.int64(0)

Because the purpose of this table is to analyze the marketing and sales funnel, all marketing leads should be preserved, including those that did not result in a closed deal.

Now the grain of `funnel_master` is: 1 row = 1 marketing lead (`mql_id`)

In [27]:
funnel_master = marketing_leads.merge(
    closed_deals,
    on="mql_id",
    how="left",
    validate="one_to_one"
)

print(funnel_master.shape)
print(funnel_master["mql_id"].nunique())

(8000, 25)
8000


After integrating `marketing_leads` and `closed_deals`, the `funnel_master` contains one row per marketing lead.

The next step was to enrich the funnel data with seller-level information from the `sellers` table. Before performing the merge, the relationship between `closed_deals` and `sellers` was checked through `seller_id`.

- `sellers` contains 3,095 unique `seller_id` values.
- `sellers.seller_id` contains no duplicates.
- `closed_deals` contains 842 unique `seller_id` values.
- 380 of the 842 seller IDs in `closed_deals` have a corresponding record in `sellers`.
- The remaining 462 seller IDs do not appear in the `sellers` table.


In [28]:
print(sellers.shape)
print(sellers["seller_id"].nunique())
print(sellers["seller_id"].duplicated().sum())

(3095, 6)
3095
0


In [29]:
print(
    closed_deals["seller_id"].isin(sellers["seller_id"]).sum()
)

380


In [30]:
print(
    closed_deals.loc[
        ~closed_deals["seller_id"].isin(sellers["seller_id"]),
        "seller_id"
    ].nunique()
)

462



I also checked these 462 unmatched seller IDs against `order_items` and obtained the same result. This indicates that these sellers do not appear in the order-level seller data either. Therefore, the absence of seller information for these records is treated as a characteristic of the available data rather than as a data-cleaning error.

In [31]:
order_seller_ids = order_items["seller_id"].unique()

print(
    closed_deals["seller_id"].isin(order_seller_ids).sum()
)

380


In [32]:
print(
    closed_deals.loc[
        ~closed_deals["seller_id"].isin(order_seller_ids),
        "seller_id"
    ].nunique()
)

462


This merge adds seller-level attributes without changing the grain of `funnel_master`.

Missing seller attributes are expected for leads that do not have a corresponding seller record. For example, `seller_city` has 7,620 missing values. These missing values should not be interpreted as a general data-cleaning problem because many marketing leads did not result in a seller/closed deal, and some closed deals do not have a matching record in the seller/order data.

These missing values will be considered later during feature preparation depending on whether the relevant seller-level feature requires seller information.

In [33]:
funnel_master = funnel_master.merge(
    sellers,
    on="seller_id",
    how="left",
    validate="many_to_one"
)

print(funnel_master.shape)
print(funnel_master["mql_id"].nunique())
print(funnel_master["seller_id"].notna().sum())

(8000, 30)
8000
842


In [34]:
funnel_master["seller_city"].isna().sum()

np.int64(7620)

ŞURAYA BAK

In [35]:
df_master.isna().sum().sort_values(ascending=False)

review_comment_title             87889
review_comment_message           58665
order_delivered_customer_date     2965
estimated_delivery_gap_days       2965
delivery_time_days                2965
shipping_time_days                1797
order_delivered_carrier_date      1783
order_total_price                  775
order_total_freight                775
order_total_cost                   775
sellers_in_order                   775
is_free_shipping                   775
average_item_price                 775
items_in_order                     775
review_length                      768
review_score                       768
review_creation_date               768
review_answer_timestamp            768
response_time_hours                768
has_comment                        768
review_id                          768
approval_time_hours                160
order_approved_at                  160
payment_count                        1
total_payment_value                  1
order_id                 

In [36]:
print("Shape:", funnel_master.shape)
print("Unique leads:", funnel_master["mql_id"].nunique())

Shape: (8000, 30)
Unique leads: 8000


In [37]:
funnel_master.isna().sum().sort_values(ascending=False)

has_company                      7937
has_gtin                         7936
average_stock                    7934
declared_product_catalog_size    7931
seller_region                    7620
seller_state                     7620
seller_city                      7620
seller_zip_code_prefix           7620
seller_order_count               7620
lead_behaviour_profile           7335
business_type                    7168
lead_type                        7164
business_segment                 7159
sdr_id                           7158
won_date                         7158
seller_id                        7158
sr_id                            7158
declared_monthly_revenue         7158
won_year                         7158
won_month                        7158
won_quarter                      7158
has_declared_revenue             7158
origin                             60
contact_year                        0
landing_page_id                     0
contact_quarter                     0
contact_mont

The `products` table's grain is different from the `df_master` table's: "1 row = 1 product". Therefore, the `products` table cannot be directly integrated into the `df_master` table. A single order can contain multiple products, so directly joining product-level data duplicates order-level records and changes the level of detail in the `df_master` table.

In [38]:
print(products.shape)
print(products.columns.tolist())
print(products["product_id"].nunique())
print(products["product_id"].duplicated().sum())

(32951, 12)
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_volume_cm3', 'product_density_g_cm3', 'has_complete_listing']
32951
0


Before integration, I checked the relationship between `order_items` and `products`. All 32,951 different products in the `order_items` table have a corresponding record in the `products` table.

In [39]:
print(
    order_items["product_id"].isin(products["product_id"]).sum()
)

print(
    order_items["product_id"].nunique()
)

112650
32951


First, I merged the order_items and products tables. The purpose of this merge was to enrich each order item with relevant product attributes while preserving the original order item detail level. The merge did not duplicate order items.

In [40]:
order_items_products = order_items.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one"
)

In [41]:
print(order_items_products.shape)
print(order_items_products["order_id"].nunique())

(112650, 27)
98666


These processes allow for the safe insertion of product-level information into order-level specifications without altering the level of detail in the df_master table, by converting product-level information into order-level specifications.

In [42]:
order_product_summary = (
    order_items_products
    .groupby("order_id")
    .agg(
        avg_product_weight_g=("product_weight_g", "mean"),
        avg_product_volume_cm3=("product_volume_cm3", "mean"),
        avg_product_density_g_cm3=("product_density_g_cm3", "mean"),
        avg_product_photos_qty=("product_photos_qty", "mean"),
        avg_product_name_length=("product_name_lenght", "mean"),
        avg_product_description_length=("product_description_lenght", "mean"),
        complete_listing_ratio=("has_complete_listing", "mean"),
        unique_product_categories=("product_category_name", "nunique")
    )
    .reset_index()
)

The resulting `order_product_summary` table shows:

1 row = 1 order.

The number of rows in the `df_master` table remained unchanged, confirming that the level of detail at the order level was preserved.

Why are the original product-level attributes still preserved?

The product-level attributes previously created in the `products` table are not replaced with these order-level summaries. Table and attribute selection should depend on the grain size being examined and the analytical question. They represent different analytical levels:

`products` → defines the attributes of an individual product
`df_master` → defines the overall product attributes of an order

In [43]:
df_master = df_master.merge(
    order_product_summary,
    on="order_id",
    how="left",
    validate="one_to_one"
)

Customer-level behavioral attributes are generated by aggregating order-level information with `customer_unique_id`.

The goal is to move from the order level to the customer level and capture the overall purchasing behavior of each customer.

Unlike order-level attributes, these variables define the customer's history across their orders. Therefore, each customer should only appear once in the resulting `customer_summary`.

The aggregation process is performed across existing order data, rather than filtering only delivered orders. This preserves customers with orders of varying statuses and prevents customers from being excluded from the customer-level population.

For monetary attributes, orders without price information are not considered zero spending. Missing transaction values ​​remain incomplete when the underlying information is unavailable.

In [44]:
customer_summary = (
    df_master
    .groupby("customer_unique_id")
    .agg(
        customer_order_count=("order_id", "nunique"),

        customer_total_spend=(
            "order_total_price",
            lambda x: x.sum(min_count=1)
        ),

        customer_avg_order_value=(
            "order_total_price",
            "mean"
        ),

        customer_total_freight=(
            "order_total_freight",
            lambda x: x.sum(min_count=1)
        ),

        customer_avg_freight=(
            "order_total_freight",
            "mean"
        ),

        customer_first_order_date=(
            "order_purchase_timestamp",
            "min"
        ),

        customer_last_order_date=(
            "order_purchase_timestamp",
            "max"
        )
    )
    .reset_index()
)

print(customer_summary.shape)
print(customer_summary["customer_unique_id"].nunique())

(96096, 8)
96096


In [47]:
order_items_products_customer = (
    order_items_products
    .merge(
        orders[["order_id", "customer_id"]],
        on="order_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        customers[["customer_id", "customer_unique_id"]],
        on="customer_id",
        how="left",
        validate="many_to_one"
    )
)

In addition to purchase frequency and monetary behavior, product variety is also recorded at the customer level.

`customer_unique_categories` represents the number of different product categories purchased by each customer.

This feature is calculated separately because product category information comes from order item/product level data, not directly from the order table.

The result is aggregated by customer level so that it can be securely integrated with `customer_summary`.

In [48]:
customer_category_summary = (
    order_items_products_customer
    .groupby("customer_unique_id")
    .agg(
        customer_unique_categories=(
            "product_category_name",
            "nunique"
        )
    )
    .reset_index()
)

print(customer_category_summary.shape)
print(
    customer_category_summary["customer_unique_id"].nunique()
)

(95420, 2)
95420


In [ ]:
customer_summary = customer_summary.merge(
    customer_category_summary,
    on="customer_unique_id",
    how="left",
    validate="one_to_one"
)

In [ ]:
print(customer_summary.shape)
print(customer_summary["customer_unique_id"].nunique())

print(
    customer_summary[
        [
            "customer_order_count",
            "customer_total_spend",
            "customer_avg_order_value",
            "customer_total_freight",
            "customer_avg_freight",
            "customer_unique_categories"
        ]
    ].isna().sum()
)

(96096, 9)
96096
customer_order_count            0
customer_total_spend          676
customer_avg_order_value      676
customer_total_freight        676
customer_avg_freight          676
customer_unique_categories    676
dtype: int64


The finalized customer-level feature table is merged back into `df_master` using `customer_unique_id`.

Before the merge, the previous versions of the customer-level features are removed to prevent duplicate columns and `_x` / `_y` suffixes.

In [ ]:
customer_features = [
    "customer_order_count",
    "customer_total_spend",
    "customer_avg_order_value",
    "customer_total_freight",
    "customer_avg_freight",
    "customer_unique_categories"
]

df_master = df_master.drop(
    columns=customer_features,
    errors="ignore"
)

df_master = df_master.merge(
    customer_summary,
    on="customer_unique_id",
    how="left",
    validate="many_to_one"
)

print("df_master shape:", df_master.shape)
print("Unique orders:", df_master["order_id"].nunique())
print("Unique customers:", df_master["customer_unique_id"].nunique())

df_master shape: (99441, 62)
Unique orders: 99441
Unique customers: 96096


After integrating the customer-level features, the master dataset is validated to ensure that:

- The order-level grain is preserved.
- No orders were duplicated or lost.
- The number of unique customers remains unchanged.
- Customer-level features have been correctly propagated to each order belonging to the same customer.
- Missing values are inspected and interpreted according to the availability of the underlying transactional data.

Missing customer-level monetary features are not automatically imputed because a missing value indicates unavailable transactional information rather than zero customer activity.

In [ ]:
customer_features = [
    "customer_order_count",
    "customer_total_spend",
    "customer_avg_order_value",
    "customer_total_freight",
    "customer_avg_freight",
    "customer_unique_categories"
]

print(
    df_master[customer_features].isna().sum()
)

customer_order_count            0
customer_total_spend          685
customer_avg_order_value      685
customer_total_freight        685
customer_avg_freight          685
customer_unique_categories    685
dtype: int64


In [ ]:
print(
    df_master[
        [
            "order_id",
            "customer_unique_id",
            "customer_order_count",
            "customer_total_spend",
            "customer_avg_order_value",
            "customer_unique_categories"
        ]
    ].head(10)
)

                           order_id                customer_unique_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  7c396fd4830fd04220f754e42b4e5bff   
1  53cdb2fc8bc7dce0b6741e2150273451  af07308b275d755c9edb36a90c618231   
2  47770eb9100c2d0c44946d9cf07ec65d  3a653a41f6f9fc3d2a113cf8398680e8   
3  949d5b44dbf5de918fe9c16f97b45f8a  7c142cf63193a1473d2e66489a9ae977   
4  ad21c59c0840e6cb83a9ceb5573f8159  72632f0f9dd73dfee390c9b22eb56dd6   
5  a4591c265e18cb1dcee52889e2d8acc3  80bb27c7c16e8f973207a5086ab329e2   
6  136cce7faa42fdb2cefd53fdc79a6098  36edbb3fb164b1f16485364b6fb04c73   
7  6514b8ad8028c9f2cc2374ded245783f  932afa1e708222e5821dac9cd5db4cae   
8  76c6e866289321a7c93b82b54852dc33  39382392765b6dc74812866ee5ee92a7   
9  e69bfb5eb88e0ed6a785585b27e16dbf  299905e3934e9e181bfb2e164dd4b4f8   

   customer_order_count  customer_total_spend  customer_avg_order_value  \
0                     2                 65.38                     32.69   
1                     1                118.70 

The base grain of the final `df_master` table must be maintained as **1 row = 1 sequence**. I have performed the necessary checks for this. This was done to ensure that all one-to-one/many-to-one join operations do not corrupt the grain at the sequence level.

In [ ]:
print("df_master shape:", df_master.shape)
print("Unique orders:", df_master["order_id"].nunique())
print("Duplicate order_id:", df_master["order_id"].duplicated().sum())
print("Missing order_id:", df_master["order_id"].isna().sum())

df_master shape: (99441, 62)
Unique orders: 99441
Duplicate order_id: 0
Missing order_id: 0


The data types of the variables in the final master table have been checked. Data types have not yet been changed at this stage. Variables that appear problematic as a result of the audit will be addressed in the next data preparation phase.

In [ ]:
df_master.dtypes

order_id                           str
customer_id                        str
order_status                       str
order_purchase_timestamp           str
order_approved_at                  str
                                ...   
customer_total_freight         float64
customer_avg_freight           float64
customer_first_order_date_y        str
customer_last_order_date_y         str
customer_unique_categories     float64
Length: 62, dtype: object

The missing value ratios of all features have been checked in the final `df_master`. Missing values ​​were neither deleted nor filled in at this stage. This is because not all missing values ​​have the same meaning.

For example:

- Empty review/comment fields → the customer hasn't written a review.

- Empty delivery date → the order hasn't been delivered.

- Empty order item summary features → there are no item records for that order.

- Empty customer/seller/funnel features → the relevant entity is not associated with that order/customer.

In [ ]:
missing_audit = (
    df_master.isna()
    .sum()
    .reset_index()
)

missing_audit.columns = ["feature", "missing_count"]

missing_audit["missing_pct"] = (
    missing_audit["missing_count"]
    / len(df_master)
    * 100
)

missing_audit = missing_audit.sort_values(
    "missing_count",
    ascending=False
)

missing_audit

,feature,missing_count,missing_pct
37,review_comment_title,87889,88.383061
38,review_comment_message,58665,58.994781
17,estimated_delivery_gap_days,2965,2.981668
6,order_delivered_customer_date,2965,2.981668
16,delivery_time_days,2965,2.981668
...,...,...,...
23,customer_state,0,0.000000
24,is_repeat_customer,0,0.000000
25,customer_region,0,0.000000
1,customer_id,0,0.000000


Basic descriptive statistics for the numeric variables in the final master table have been examined.

The purpose of this check is to identify unexpected negative values, extreme values, unexpected ranges, and general distributions of features.

Outlier or abnormal values ​​have not yet been deleted at this stage. First, it will be investigated whether these values ​​originate from the data generation process or are actual observations.

In [ ]:
numeric_cols = df_master.select_dtypes(
    include=["int64", "float64"]
).columns

df_master[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
purchase_year,99441.0,2017.539838,0.505007,2016.000000,2017.000000,2018.000000,2018.000000,2018.000000
purchase_month,99441.0,6.032220,3.232999,1.000000,3.000000,6.000000,8.000000,12.000000
purchase_day,99441.0,15.505948,8.667298,1.000000,8.000000,15.000000,23.000000,31.000000
purchase_hour,99441.0,14.770829,5.326800,0.000000,11.000000,15.000000,19.000000,23.000000
approval_time_hours,99281.0,10.419094,26.038004,0.000000,0.215000,0.343333,14.580833,4509.180556
shipping_time_days,97644.0,2.805038,3.549427,-171.219005,0.875509,1.818397,3.580469,125.762569
delivery_time_days,96476.0,12.558702,9.546530,0.533414,6.766403,10.217755,15.720327,209.628611
estimated_delivery_gap_days,96476.0,-11.179120,10.186113,-146.016123,-16.244384,-11.948941,-6.390000,188.975081
customer_zip_code_prefix,99441.0,35137.474583,29797.938996,1003.000000,11347.000000,24416.000000,58900.000000,99990.000000
items_in_order,98666.0,1.141731,0.538452,1.000000,1.000000,1.000000,1.000000,21.000000


At this stage, we added the summary features we created on a customer basis to `df_master` and checked and cleaned up any duplicate columns that might occur as a result of merge operations.

When a column already exists in `df_master` and is merged again with the same name, Pandas adds suffixes to the column names to prevent conflicts:

- `_x` → column from the left/original DataFrame
- `_y` → column from the merged DataFrame

For example:

`customer_first_order_date_x`

`customer_first_order_date_y`

These suffixes are not a real feature; they are technical column names created during the merge.

In [ ]:
[c for c in df_master.columns if "_x" in c or "_y" in c]

['purchase_year',
 'customer_first_order_date_x',
 'customer_last_order_date_x',
 'customer_first_order_date_y',
 'customer_last_order_date_y']

In [ ]:
[c for c in df_master.columns if "customer_" in c]

['customer_id',
 'order_delivered_customer_date',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'customer_region',
 'customer_first_order_date_x',
 'customer_last_order_date_x',
 'customer_order_count',
 'customer_total_spend',
 'customer_avg_order_value',
 'customer_total_freight',
 'customer_avg_freight',
 'customer_first_order_date_y',
 'customer_last_order_date_y',
 'customer_unique_categories']

Therefore, we compared the values ​​of the two columns. The results showed that the `_x` and `_y` values ​​were largely the same, but in some rows the `_x` side was missing (`NaN`).

In this case, the `_y` columns were kept because the `_y` version was more complete.

Duplicate `_x` columns were removed. Then, `_y` columns were returned to their final names:

This eliminated technical duplicate columns representing the same information within the dataset.

The dataset now contains only one clean column.

This process was not only done to clean up column names; it was also done to prevent the accidental use of an old/incomplete version as a result of the merge.

In [ ]:
(df_master["customer_first_order_date_x"] != 
 df_master["customer_first_order_date_y"]).sum()

np.int64(2964)

In [ ]:
(df_master["customer_last_order_date_x"] != 
 df_master["customer_last_order_date_y"]).sum()

np.int64(2917)

In [ ]:
duplicate_columns = df_master.columns[df_master.columns.duplicated()].tolist()

print("Duplicate columns:", duplicate_columns)

Duplicate columns: []


In [ ]:
xy_columns = [
    col for col in df_master.columns
    if col.endswith("_x") or col.endswith("_y")
]

print(xy_columns)

['customer_first_order_date_x', 'customer_last_order_date_x', 'customer_first_order_date_y', 'customer_last_order_date_y']


In [ ]:
df_master[
    [
        "customer_first_order_date_x",
        "customer_first_order_date_y",
        "customer_last_order_date_x",
        "customer_last_order_date_y"
    ]
].head(20)

,customer_first_order_date_x,customer_first_order_date_y,customer_last_order_date_x,customer_last_order_date_y
0,2017-09-04 11:26:38,2017-09-04 11:26:38,2017-10-02 10:56:33,2017-10-02 10:56:33
1,2018-07-24 20:41:37,2018-07-24 20:41:37,2018-07-24 20:41:37,2018-07-24 20:41:37
2,2018-08-08 08:38:49,2018-08-08 08:38:49,2018-08-08 08:38:49,2018-08-08 08:38:49
3,2017-11-18 19:28:06,2017-11-18 19:28:06,2017-11-18 19:28:06,2017-11-18 19:28:06
4,2018-02-13 21:18:39,2018-02-13 21:18:39,2018-02-13 21:18:39,2018-02-13 21:18:39
5,2017-07-09 21:57:05,2017-07-09 21:57:05,2017-07-09 21:57:05,2017-07-09 21:57:05
6,NaN,2017-04-11 12:22:08,NaN,2017-04-11 12:22:08
7,2017-05-16 13:10:30,2017-05-16 13:10:30,2017-05-16 13:10:30,2017-05-16 13:10:30
8,2017-01-23 18:29:09,2017-01-23 18:29:09,2017-01-23 18:29:09,2017-01-23 18:29:09
9,2017-07-29 11:55:02,2017-07-29 11:55:02,2017-07-29 11:55:02,2017-07-29 11:55:02


As a result, `df_master` has become a richer analytics table that carries both order-level information and customer
past behavior.

In [ ]:
df_master = df_master.drop(columns=[
    "customer_first_order_date_x",
    "customer_last_order_date_x"
])

df_master = df_master.rename(columns={
    "customer_first_order_date_y": "customer_first_order_date",
    "customer_last_order_date_y": "customer_last_order_date"
})

In [ ]:
print([
    col for col in df_master.columns
    if "customer_first_order_date" in col
    or "customer_last_order_date" in col
])

['customer_first_order_date', 'customer_last_order_date']


I checked the overall structure and the duplicate status. There are no problems at this stage.

In [ ]:
print("Shape:", df_master.shape)

print("Duplicate rows:", df_master.duplicated().sum())
print("Duplicate order_id:", df_master["order_id"].duplicated().sum())
print("Missing order_id:", df_master["order_id"].isna().sum())

print("\nDuplicate column names:")
print(df_master.columns[df_master.columns.duplicated()].tolist())

Shape: (99441, 60)
Duplicate rows: 0
Duplicate order_id: 0
Missing order_id: 0

Duplicate column names:
[]


I checked for missing values ​​in the critical ID columns. There doesn't seem to be a problem there either.

In [ ]:
id_cols = [
    "order_id",
    "customer_id",
    "customer_unique_id"
]

print(df_master[id_cols].isna().sum())

order_id              0
customer_id           0
customer_unique_id    0
dtype: int64


I'm observing if there are any negative values, and if so, in which columns they are present.

The negative values ​​for estimated_delivery_gap_days are normal because I created the feature accordingly. I used the formula "actual delivery date - estimated delivery date". Therefore, negative values ​​actually represent "early delivery". That's why I'm not touching it. This could be a deliberate business strategy.

In [ ]:
numeric_cols = df_master.select_dtypes(include="number").columns

negative_counts = (df_master[numeric_cols] < 0).sum()

print(negative_counts[negative_counts > 0].sort_values(ascending=False))

estimated_delivery_gap_days    88649
shipping_time_days              1359
dtype: int64


I'm not deleting the `shipping_time_days` value. Instead, I'm creating an anomaly flag. This way, the information isn't lost, and anomalies can be addressed during the modeling phase.

In [49]:
df_master["shipping_negative_anomaly"] = (
    df_master["shipping_time_days"] < 0
)

df_master["shipping_negative_anomaly"].value_counts()

shipping_negative_anomaly
False    98082
True      1359
Name: count, dtype: int64

To identify unusually high values ​​in the `shipping_time_days` variable, I initially excluded negative shipping times from the analysis. I separately flagged negative values ​​with the `shipping_negative_anomaly` flag.

For positive shipping times, I applied the Interquartile Range (IQR) method:

- Q1 = 0.90 days
- Q3 = 3.62 days
- IQR = 2.72 days
- Upper limit = Q3 + 1.5 × IQR = 7.71 days

Therefore, I marked shipping times longer than 7.71 days as `shipping_long_anomaly = True`.

In [50]:
shipping_clean = df_master.loc[
    ~df_master["shipping_negative_anomaly"],
    "shipping_time_days"
]

Q1 = shipping_clean.quantile(0.25)
Q3 = shipping_clean.quantile(0.75)
IQR = Q3 - Q1

upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Upper bound:", upper_bound)

Q1: 0.901539351851852
Q3: 3.624050925925926
IQR: 2.7225115740740744
Upper bound: 7.707818287037037


I created this not to erase these flag values, but to allow for the separate evaluation of unusually long delivery/carrier processes in future statistical analyses and reviews of logistics operation performance.

In [51]:
df_master["shipping_long_anomaly"] = (
    df_master["shipping_time_days"] > upper_bound
)

print(df_master["shipping_long_anomaly"].value_counts())

shipping_long_anomaly
False    93954
True      5487
Name: count, dtype: int64


In conclusion, this value corresponds to 5.62% of current shipping times. I have not considered or deleted these records as erroneous data. `shipping_long_anomaly` was created solely to statistically identify unusually long shipping times.

In [53]:
long_anomaly_count = df_master["shipping_long_anomaly"].sum()
long_anomaly_pct = (
    long_anomaly_count / df_master["shipping_time_days"].notna().sum()
) * 100

print("Long shipping anomalies:", long_anomaly_count)
print("Percentage:", round(long_anomaly_pct, 2), "%")

Long shipping anomalies: 5487
Percentage: 5.62 %


In [54]:
long_shipping = df_master[
    df_master["shipping_long_anomaly"]
].copy()

print(
    long_shipping["purchase_year"]
    .value_counts()
    .sort_index()
)

print("\nYear percentages:")
print(
    long_shipping["purchase_year"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

purchase_year
2016     165
2017    2659
2018    2663
Name: count, dtype: int64

Year percentages:
purchase_year
2016     3.01
2017    48.46
2018    48.53
Name: proportion, dtype: float64
